# 异常处理进阶 · 知识点

### 1、try/except/else/finally 完整结构

In [ ]:
# try：放可能出错的代码
# except：匹配到对应异常类型时执行
# else：try 块没有抛出任何异常时才执行（很多人不知道这个子句，容易被忽略）
# finally：无论有没有异常，最后一定会执行，常用来释放资源（关文件、关连接）

def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError as e:
        print(f"除零错误：{e}")
    else:
        print(f"计算成功，结果是 {result}")   # 只有没异常才会走到这里
    finally:
        print("除法操作结束")                  # 不管成不成功都会打印

divide(10, 2)
print("---")
divide(10, 0)

### 2、异常捕获顺序与类型层级（复习巩固）

In [ ]:
# except 要按"从具体到笼统"的顺序写，Python 按顺序匹配，第一个匹配上的生效
# 异常是有继承关系的：FileNotFoundError、PermissionError 都是 OSError 的子类，OSError 又是 Exception 的子类
# 如果把笼统的写前面，下面更具体的分支永远不会被触发

# 用元组可以一次捕获多种异常，处理逻辑相同时不用写多个 except
try:
    int("abc")
except (ValueError, TypeError) as e:
    print(f"类型或取值错误：{e}")

# 异常对象本身可以取出信息：
try:
    int("abc")
except ValueError as e:
    print(e)          # invalid literal for int() with base 10: 'abc'
    print(e.args)     # ("invalid literal for int() with base 10: 'abc'",)  参数都在这个元组里

### 3、主动抛出异常：raise

In [ ]:
# raise 异常类型(参数)：主动抛出一个异常，跟 Java 的 throw 类似
# 区别：Python 没有 checked exception，函数签名不用声明会抛什么异常（不用写 throws）
def check_age(age):
    if age < 0:
        raise ValueError(f"年龄不能为负数：{age}")
    return age

try:
    check_age(-1)
except ValueError as e:
    print(f"捕获到：{e}")

In [ ]:
# except 块里单独写 raise（不带任何参数）：把当前捕获到的异常原样重新抛出
# 常见场景：想在中间层记一下日志，但异常还是要继续往上层传，不能就这么"吃掉"
def process():
    try:
        int("abc")
    except ValueError as e:
        print(f"记录日志：处理失败 - {e}")
        raise   # 重新抛出，调用方依然能捕获到同一个异常

try:
    process()
except ValueError as e:
    print(f"上层捕获到：{e}")

### 4、自定义异常类

In [ ]:
# 自定义异常：继承 Exception（或更具体的内置异常），可以加自己的属性，让错误信息更贴近业务
# 对比 Java：跟 自定义 Exception 继承 RuntimeException/Exception 的写法基本一样
class InsufficientBalanceError(Exception):
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"余额不足：当前余额 {balance}，需要 {amount}")

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientBalanceError(balance, amount)
    return balance - amount

try:
    withdraw(100, 200)
except InsufficientBalanceError as e:
    print(e)                  # 余额不足：当前余额 100，需要 200
    print(e.balance, e.amount)  # 自定义属性也能直接取，比单纯一个字符串信息更有用

### 5、异常链：raise ... from

In [3]:
# 场景：捕获了一个底层异常（比如解析失败），想抛出一个更贴合业务语义的新异常，
# 但又不想丢掉原始异常的信息——这时候用 raise NewException(...) from original_exception

class ConfigError(Exception):
    pass

def load_config(text):
    import json
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ConfigError("配置文件格式不对，加载失败") from e

try:
    load_config("not a json")
except ConfigError as e:
    print(e)                # 配置文件格式不对，加载失败
    print(e.__cause__)      # 原始的 JSONDecodeError，信息没有丢

# 对比：如果不写 from，Python 打印的提示是
# "During handling of the above exception, another exception occurred"（两个异常看起来不相关）
# 写了 from 之后提示是
# "The above exception was the direct cause of the following exception"（明确告诉你因果关系）

配置文件格式不对，加载失败
Expecting value: line 1 column 1 (char 0)


### 6、with 语句与上下文管理器协议

In [4]:
# with 语句背后的协议：对象要实现 __enter__ 和 __exit__ 两个方法
# __enter__(self)：进入 with 块之前调用，返回值会赋给 as 后面的变量
# __exit__(self, exc_type, exc_value, traceback)：离开 with 块时调用，不管 with 块里有没有异常都会调
#   - 三个参数：如果块内抛了异常，分别是异常类型/异常对象/traceback；没异常这三个全是 None
#   - 返回 True：表示这个异常已经被"处理"了，不会再往外传播（用得少，要谨慎用）
#   - 返回 False（或不写 return，默认 None）：异常会正常继续往外抛

# open() 返回的文件对象就实现了这个协议，这也是为什么可以 with open(...) as f
# 对比 Java：相当于 try-with-resources + Closeable.close()，但 __exit__ 能拿到异常信息并决定要不要吞掉它，比 close() 更灵活
print(hasattr(open("knowledge_demo.txt", "w"), "__enter__"))

True


### 7、自定义上下文管理器类

In [ ]:
import time

# 自己实现 __enter__/__exit__，可以把"进入/退出某个代码块"这件事封装起来，常用于计时、资源管理
class Timer:
    def __enter__(self):
        self.start = time.time()
        return self   # 返回的对象会赋给 as 后面的变量

    def __exit__(self, exc_type, exc_value, traceback):
        elapsed = time.time() - self.start
        print(f"耗时：{elapsed:.4f} 秒")
        return False   # 不吞异常，块内如果出错该报还是报

with Timer() as t:
    total = sum(range(1000000))
print(total)

### 8、用 contextlib.contextmanager 简化

In [5]:
from contextlib import contextmanager
import time

# 用一个生成器函数 + @contextmanager 装饰器，比写一整个类更简洁
# yield 之前的代码 = __enter__；yield 后面带的值 = with ... as 变量 接到的值（跟类写法里 __enter__ 的 return 是同一个机制）；yield 之后的代码 = __exit__
# 如果要保证"即使块内出错也执行收尾代码"，要用 try/finally 包住 yield（跟 __exit__ 总会执行是同一个道理）
@contextmanager
def timer():
    start = time.time()
    try:
        yield start
    finally:
        elapsed = time.time() - start
        print(f"耗时：{elapsed:.4f} 秒")

with timer() as start_time:        # 用 as 接住 yield 出来的值，块内可以直接用
    total = sum(range(1000000))
    print(f"开始时间戳：{start_time}")
print(total)

开始时间戳：1782201573.3610082
耗时：0.0224 秒
499999500000


### 9、contextlib 常用工具

In [ ]:
from contextlib import suppress

# suppress(异常类型)：忽略指定异常，等价于 try/except 但只有一行，避免写一堆空的 except: pass
with suppress(FileNotFoundError):
    open("not_exist.txt").read()
print("继续往下执行，没有被异常打断")

# 对比写法：
# try:
#     open("not_exist.txt").read()
# except FileNotFoundError:
#     pass